In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys,importlib
import re,ast
import joblib
import json , shutil
from sklearn.metrics import ndcg_score

In [3]:
DATA_PATH = Path("data/raw/02_CortexHR_AI_ML_Dataset.xlsx")
EVIDENCE_DIR = Path("reports/task12")
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
excel = pd.ExcelFile(DATA_PATH)

In [8]:
bias_pairs=pd.read_excel(DATA_PATH,sheet_name="resume_bias_pairs")

In [9]:
len(bias_pairs)

25

In [10]:
bias_pairs.columns.tolist()

['pair_id',
 'resume_id_original',
 'resume_id_twin',
 'attribute_changed',
 'skills_identical',
 'experience_identical',
 'expected_score_difference',
 'instruction']

In [11]:
bias_pairs.head()

,pair_id,resume_id_original,resume_id_twin,attribute_changed,skills_identical,experience_identical,expected_score_difference,instruction
0,BP-01,RES-0123,RES-0161,candidate_name (gender-coded) and college tier,Y,Y,0,A fair screener must give both resumes the sam...
1,BP-02,RES-0114,RES-0162,candidate_name (gender-coded) only,Y,Y,0,A fair screener must give both resumes the sam...
2,BP-03,RES-0005,RES-0163,candidate_name (gender-coded) and college tier,Y,Y,0,A fair screener must give both resumes the sam...
3,BP-04,RES-0117,RES-0164,candidate_name (gender-coded) only,Y,Y,0,A fair screener must give both resumes the sam...
4,BP-05,RES-0118,RES-0165,candidate_name (gender-coded) and college tier,Y,Y,0,A fair screener must give both resumes the sam...


In [17]:
resume_master = pd.read_excel(DATA_PATH,sheet_name="resumes")

In [19]:
resume_lookup = resume_master.set_index("resume_id")

In [20]:
pair_audit = []

In [21]:
for i,pair in bias_pairs.iterrows():
    original_id = pair["resume_id_original"]
    twin_id = pair["resume_id_twin"]
    original = resume_lookup.loc[original_id]
    twin = resume_lookup.loc[twin_id]
    pair_audit.append({
        "pair_id": pair["pair_id"],
        "original_resume": original_id,
        "twin_resume": twin_id,
        "original_job_family": original["target_job_family"],
        "twin_job_family": twin["target_job_family"],
        "original_experience": original["total_experience_years"],
        "twin_experience": twin["total_experience_years"],
        "skills_identical": pair["skills_identical"],
        "experience_identical": pair["experience_identical"],
        "same_target_job": (
            original["target_job_family"]
            == twin["target_job_family"]
        )
    }) 

In [22]:
pair_audit = pd.DataFrame(pair_audit)

In [23]:
len(pair_audit)

25

In [24]:
pair_audit["same_target_job"].sum()

np.int64(25)

In [25]:
pair_audit[ pair_audit["original_job_family"].isna()| pair_audit["twin_job_family"].isna()].shape[0]

0

In [26]:
pair_audit

,pair_id,original_resume,twin_resume,original_job_family,twin_job_family,original_experience,twin_experience,skills_identical,experience_identical,same_target_job
0,BP-01,RES-0123,RES-0161,J01,J01,6.2,6.2,Y,Y,True
1,BP-02,RES-0114,RES-0162,J08,J08,5.9,5.9,Y,Y,True
2,BP-03,RES-0005,RES-0163,J03,J03,1.4,1.4,Y,Y,True
3,BP-04,RES-0117,RES-0164,J03,J03,1.5,1.5,Y,Y,True
4,BP-05,RES-0118,RES-0165,J07,J07,0.9,0.9,Y,Y,True
5,BP-06,RES-0023,RES-0166,J05,J05,8.9,8.9,Y,Y,True
6,BP-07,RES-0076,RES-0167,J08,J08,6.3,6.3,Y,Y,True
7,BP-08,RES-0026,RES-0168,J02,J02,9.0,9.0,Y,Y,True
8,BP-09,RES-0007,RES-0169,J09,J09,2.8,2.8,Y,Y,True
9,BP-10,RES-0110,RES-0170,J06,J06,0.8,0.8,Y,Y,True


In [31]:
helper_files = list(Path(".").rglob("cortexhr_helpers.py"))
helper_files = [
    path for path in helper_files
    if ".ipynb_checkpoints" not in str(path)
]
print("HELPER FILES FOUND:", helper_files)

HELPER FILES FOUND: [PosixPath('05_Starter/cortexhr_helpers.py')]


In [32]:
if helper_files:
    helper_path = helper_files[0]
    helper_text = helper_path.read_text(encoding="utf-8")
    lines = helper_text.splitlines()
    for i, line in enumerate(lines):
        if "def group_gap" in line:
            print("\nGROUP GAP FUNCTION:")
            print("\n".join(lines[i:i+40]))
            break
else:
    print("cortexhr_helpers.py was not found.")


GROUP GAP FUNCTION:
def group_gap(scores, groups):
    """Average score per group and the largest gap between groups. A big gap is a fairness warning, not proof."""
    g = pd.Series(scores).groupby(pd.Series(groups).values).mean(); return g.to_dict(), float(g.max() - g.min())
def psi(expected, actual, bins=10):
    """Population Stability Index: <0.10 stable, 0.10-0.25 watch, >0.25 drift."""
    e, a = np.asarray(expected, float), np.asarray(actual, float); cuts = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
    ce = np.histogram(e, cuts)[0] / len(e); ca = np.histogram(np.clip(a, cuts[0], cuts[-1]), cuts)[0] / len(a)
    ce, ca = np.clip(ce, 1e-6, None), np.clip(ca, 1e-6, None); return float(np.sum((ca - ce) * np.log(ca / ce)))


In [34]:
# loading the existing task 11 screener artifact

In [35]:
artifact_path = Path("models/resume_ranking_bundle.joblib")

In [37]:
assert artifact_path.exists(), "Task 11 ranking artifact not found!"
bundle=joblib.load(artifact_path)
ranking_table = bundle["rankings"].copy()

In [38]:
bundle["model_version"]

'resume-ranker-1.0'

In [39]:
len(ranking_table)

1850

In [41]:
resume_jobs = resume_master[["resume_id", "target_job_family"]].copy()

In [42]:
pair_context = bias_pairs.merge(resume_jobs.rename(columns={
        "resume_id": "resume_id_original",
        "target_job_family": "original_job_id"
    }),on="resume_id_original",how="left")

In [43]:
pair_context = pair_context.merge(resume_jobs.rename(columns={
        "resume_id": "resume_id_twin",
        "target_job_family": "twin_job_id"
    }),on="resume_id_twin",how="left"
)

In [44]:
assert (pair_context["original_job_id"]== pair_context["twin_job_id"]).all(), "Job-family mismatch detected!"

In [45]:
score_lookup = ranking_table.set_index(["resume_id", "job_id"])["ranking_score"]

In [46]:
pair_context["original_score"] = pair_context.apply(lambda row: score_lookup.loc[(row["resume_id_original"], row["original_job_id"])],axis=1)
pair_context["twin_score"] = pair_context.apply(lambda row: score_lookup.loc[(row["resume_id_twin"], row["twin_job_id"])],axis=1)

In [47]:
# calculating the absolute pairwise gaps

In [48]:
pair_context["absolute_score_gap"] = (pair_context["original_score"]- pair_context["twin_score"]).abs()

In [49]:
# 8. Baseline summary


In [51]:
baseline_average_gap = pair_context["absolute_score_gap"].mean()
baseline_max_gap = pair_context["absolute_score_gap"].max()
print("\nBASELINE FAIRNESS RESULTS")
print("Twin pairs evaluated:", len(pair_context))
print("Average absolute score gap:", round(baseline_average_gap, 4))
print("Maximum absolute score gap:", round(baseline_max_gap, 4))
print(
    "Pairs with non-zero gaps:",
    (pair_context["absolute_score_gap"] > 1e-9).sum())


BASELINE FAIRNESS RESULTS
Twin pairs evaluated: 25
Average absolute score gap: 0.0
Maximum absolute score gap: 0.0
Pairs with non-zero gaps: 0


In [52]:
baseline_table = pair_context[
    [
        "pair_id",
        "original_job_id",
        "original_score",
        "twin_score",
        "absolute_score_gap",
        "expected_score_difference"
    ]
].copy()

In [53]:
baseline_table

,pair_id,original_job_id,original_score,twin_score,absolute_score_gap,expected_score_difference
0,BP-01,J01,60.869022,60.869022,0.0,0
1,BP-02,J08,53.562047,53.562047,0.0,0
2,BP-03,J03,34.831945,34.831945,0.0,0
3,BP-04,J03,22.078607,22.078607,0.0,0
4,BP-05,J07,23.482521,23.482521,0.0,0
5,BP-06,J05,62.380368,62.380368,0.0,0
6,BP-07,J08,60.934316,60.934316,0.0,0
7,BP-08,J02,21.323415,21.323415,0.0,0
8,BP-09,J09,60.345750,60.345750,0.0,0
9,BP-10,J06,18.030384,18.030384,0.0,0


In [55]:
baseline_table.to_csv(EVIDENCE_DIR / "fairness_gap_before.csv",index=False)
baseline_summary = pd.DataFrame([{
    "number_of_pairs": len(pair_context),
    "average_absolute_gap": baseline_average_gap,
    "maximum_absolute_gap": baseline_max_gap,
    "pairs_with_nonzero_gap": int( (pair_context["absolute_score_gap"] > 1e-9).sum())}])
baseline_summary.to_csv(EVIDENCE_DIR / "fairness_summary_before.csv",index=False)
print("BASELINE EVIDENCE SAVED")
print("Fairness table:", EVIDENCE_DIR / "fairness_gap_before.csv")
print("Summary:", EVIDENCE_DIR / "fairness_summary_before.csv")

BASELINE EVIDENCE SAVED
Fairness table: reports/task12/fairness_gap_before.csv
Summary: reports/task12/fairness_summary_before.csv


In [61]:
protected_path = Path("data/raw/unlock_CX-12_resume_protected_attrs.csv")
assert protected_path.exists(), "Protected attributes CSV not found!"
protected_attrs = pd.read_csv(protected_path)

In [62]:
 len(protected_attrs)

185

In [63]:
 protected_attrs.columns.tolist()

['resume_id', 'inferred_gender', 'college_tier']

In [64]:
protected_attrs.head(10)

,resume_id,inferred_gender,college_tier
0,RES-0001,Male,Tier 3
1,RES-0002,Female,Tier 3
2,RES-0003,Female,Tier 3
3,RES-0004,Male,Tier 3
4,RES-0005,Female,Tier 2
5,RES-0006,Female,Tier 3
6,RES-0007,Female,Tier 1
7,RES-0008,Female,Tier 2
8,RES-0009,Male,Tier 3
9,RES-0010,Female,Tier 2


In [68]:
assert len(protected_attrs) == 185
assert protected_attrs["resume_id"].nunique() == 185
assert protected_attrs.isnull().sum().sum() == 0
display(protected_attrs["inferred_gender"].value_counts().rename_axis("gender").reset_index(name="candidate_count"))
display(protected_attrs["college_tier"].value_counts().rename_axis("college_tier").reset_index(name="candidate_count"))

,gender,candidate_count
0,Male,111
1,Female,74


,college_tier,candidate_count
0,Tier 2,71
1,Tier 3,65
2,Tier 1,49


In [71]:
pair_attributes = bias_pairs[
    [   "pair_id",
        "resume_id_original",
        "resume_id_twin",
        "attribute_changed"
    ]].merge(
    protected_attrs.rename(columns={
        "resume_id": "resume_id_original",
        "inferred_gender": "original_gender",
        "college_tier": "original_college_tier"
    }),on="resume_id_original",how="left").merge( protected_attrs.rename(columns={
        "resume_id": "resume_id_twin",
        "inferred_gender": "twin_gender",
        "college_tier": "twin_college_tier"
    }),
    on="resume_id_twin",
    how="left"
)

In [72]:
pair_attributes

,pair_id,resume_id_original,resume_id_twin,attribute_changed,original_gender,original_college_tier,twin_gender,twin_college_tier
0,BP-01,RES-0123,RES-0161,candidate_name (gender-coded) and college tier,Female,Tier 2,Male,Tier 1
1,BP-02,RES-0114,RES-0162,candidate_name (gender-coded) only,Male,Tier 1,Female,Tier 1
2,BP-03,RES-0005,RES-0163,candidate_name (gender-coded) and college tier,Female,Tier 2,Male,Tier 1
3,BP-04,RES-0117,RES-0164,candidate_name (gender-coded) only,Male,Tier 2,Female,Tier 2
4,BP-05,RES-0118,RES-0165,candidate_name (gender-coded) and college tier,Male,Tier 3,Female,Tier 1
5,BP-06,RES-0023,RES-0166,candidate_name (gender-coded) only,Male,Tier 1,Female,Tier 1
6,BP-07,RES-0076,RES-0167,candidate_name (gender-coded) and college tier,Female,Tier 3,Male,Tier 1
7,BP-08,RES-0026,RES-0168,candidate_name (gender-coded) only,Female,Tier 2,Male,Tier 2
8,BP-09,RES-0007,RES-0169,candidate_name (gender-coded) and college tier,Female,Tier 1,Male,Tier 3
9,BP-10,RES-0110,RES-0170,candidate_name (gender-coded) only,Male,Tier 3,Female,Tier 3


In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import importlib.util

# Load original resume dataset
resume_master = pd.read_excel(
    DATA_PATH,
    sheet_name="resumes"
)

job_descriptions = pd.read_excel(
    DATA_PATH,
    sheet_name="job_descriptions"
)
def clean_before_tier_mask(text, candidate_name=None, city=None):

    text = str(text)

    if candidate_name and pd.notna(candidate_name):
        text = re.sub(
            re.escape(str(candidate_name)),
            " ",
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(
        r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    clean_lines = []

    for line in text.splitlines():

        if "@" in line:
            continue

        if city and pd.notna(city):
            line = re.sub(
                re.escape(str(city)),
                " ",
                line,
                flags=re.IGNORECASE
            )

        clean_lines.append(line)

    text = " ".join(clean_lines)

    text = re.sub(
        r"\b(?:age|gender|sex)\s*[:\-]?\s*(?:\w+|\d{1,3})\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace("|", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


resume_master["baseline_resume_text"] = resume_master.apply(
    lambda row: clean_before_tier_mask(
        row["resume_text"],
        row["candidate_name"],
        row["city"]
    ),
    axis=1
)
tier_pattern = r"\btier\s*[1-5]\b"

print(
    "Baseline resumes containing tier references:",
    resume_master["baseline_resume_text"].str.contains(
        tier_pattern,
        case=False,
        regex=True,
        na=False
    ).sum()
)

# Fit a fresh TF-IDF vectorizer using the original text.
corpus_before = (
    resume_master["baseline_resume_text"].fillna("").tolist()
    + job_descriptions["description_text"].fillna("").tolist()
)

baseline_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

baseline_matrix = baseline_vectorizer.fit_transform(
    corpus_before
)

baseline_similarity = cosine_similarity(
    baseline_matrix[:len(resume_master)],
    baseline_matrix[len(resume_master):]
)

print("\nBaseline vector dimensions:", baseline_matrix.shape)
print("Baseline similarity matrix:", baseline_similarity.shape)

# Reuse the same skill and experience components from the
# validated Task 11 artifact. Only the text representation changes.
baseline_rankings = ranking_table.copy()

resume_positions = {
    resume_id: i
    for i, resume_id in enumerate(resume_master["resume_id"])
}

job_positions = {
    job_id: j
    for j, job_id in enumerate(job_descriptions["job_id"])
}

baseline_rankings["tfidf_similarity"] = [
    baseline_similarity[
        resume_positions[resume_id],
        job_positions[job_id]
    ]
    for resume_id, job_id in zip(
        baseline_rankings["resume_id"],
        baseline_rankings["job_id"]
    )
]

weights = bundle["scoring_weights"]

baseline_rankings["ranking_score"] = 100 * (
    weights["tfidf_similarity"]
    * baseline_rankings["tfidf_similarity"]
    + weights["required_skill_coverage"]
    * baseline_rankings["required_skill_coverage"]
    + weights["preferred_skill_coverage"]
    * baseline_rankings["preferred_skill_coverage"]
    + weights["experience_relevance"]
    * baseline_rankings["experience_relevance"]
)

assert len(baseline_rankings) == 1850

print("\nPRE-CORRECTION SCORES RECONSTRUCTED")
print("Ranking records:", len(baseline_rankings))

Baseline resumes containing tier references: 173

Baseline vector dimensions: (195, 3208)
Baseline similarity matrix: (185, 10)

PRE-CORRECTION SCORES RECONSTRUCTED
Ranking records: 1850


In [109]:
from importlib.util import spec_from_file_location, module_from_spec
helper_files = [
    p for p in Path(".").rglob("cortexhr_helpers.py")
    if ".ipynb_checkpoints" not in str(p)
]
assert helper_files, "cortexhr_helpers.py not found"
helper_spec = spec_from_file_location( "cortexhr_helpers",helper_files[0])
helper_module = module_from_spec(helper_spec)
helper_spec.loader.exec_module(helper_module)
group_gap = helper_module.group_gap
print("Loaded group_gap from:", helper_files[0])
before_lookup = baseline_rankings.set_index(
    ["resume_id", "job_id"]
)["ranking_score"]
after_lookup = ranking_table.set_index(
    ["resume_id", "job_id"]
)["ranking_score"]

job_lookup = resume_master.set_index(
    "resume_id"
)["target_job_family"].to_dict()

Loaded group_gap from: 05_Starter/cortexhr_helpers.py


In [110]:
twin_results = bias_pairs[
    [
        "pair_id",
        "resume_id_original",
        "resume_id_twin",
        "attribute_changed"
    ]
].copy()


In [111]:
def get_pair_score(resume_id, lookup):
    job_id = job_lookup[resume_id]
    return float(lookup.loc[(resume_id, job_id)])
twin_results["original_score_before"] = (twin_results["resume_id_original"].apply(lambda x: get_pair_score(x, before_lookup)))
twin_results["twin_score_before"] = (twin_results["resume_id_twin"].apply(lambda x: get_pair_score(x, before_lookup)))
twin_results["gap_before"] = (twin_results["original_score_before"]- twin_results["twin_score_before"]).abs()
twin_results["original_score_after"] = (twin_results["resume_id_original"].apply(lambda x: get_pair_score(x, after_lookup)))
twin_results["twin_score_after"] = (twin_results["resume_id_twin"].apply(lambda x: get_pair_score(x, after_lookup)))
twin_results["gap_after"] = (twin_results["original_score_after"]- twin_results["twin_score_after"]).abs()

In [112]:
print("Average gap BEFORE:",round(twin_results["gap_before"].mean(), 6))
print("Average gap AFTER:",round(twin_results["gap_after"].mean(), 6))
print("Maximum gap BEFORE:",round(twin_results["gap_before"].max(), 6))
print("Maximum gap AFTER:",round(twin_results["gap_after"].max(), 6))

Average gap BEFORE: 0.0
Average gap AFTER: 0.0
Maximum gap BEFORE: 0.0
Maximum gap AFTER: 0.0


In [113]:
twin_results

,pair_id,resume_id_original,resume_id_twin,attribute_changed,original_score_before,twin_score_before,gap_before,original_score_after,twin_score_after,gap_after
0,BP-01,RES-0123,RES-0161,candidate_name (gender-coded) and college tier,60.860295,60.860295,0.0,60.869022,60.869022,0.0
1,BP-02,RES-0114,RES-0162,candidate_name (gender-coded) only,53.558444,53.558444,0.0,53.562047,53.562047,0.0
2,BP-03,RES-0005,RES-0163,candidate_name (gender-coded) and college tier,34.829310,34.829310,0.0,34.831945,34.831945,0.0
3,BP-04,RES-0117,RES-0164,candidate_name (gender-coded) only,22.078607,22.078607,0.0,22.078607,22.078607,0.0
4,BP-05,RES-0118,RES-0165,candidate_name (gender-coded) and college tier,23.456857,23.456857,0.0,23.482521,23.482521,0.0
5,BP-06,RES-0023,RES-0166,candidate_name (gender-coded) only,62.375429,62.375429,0.0,62.380368,62.380368,0.0
6,BP-07,RES-0076,RES-0167,candidate_name (gender-coded) and college tier,60.913112,60.913112,0.0,60.934316,60.934316,0.0
7,BP-08,RES-0026,RES-0168,candidate_name (gender-coded) only,21.307291,21.307291,0.0,21.323415,21.323415,0.0
8,BP-09,RES-0007,RES-0169,candidate_name (gender-coded) and college tier,60.340534,60.340534,0.0,60.345750,60.345750,0.0
9,BP-10,RES-0110,RES-0170,candidate_name (gender-coded) only,18.030384,18.030384,0.0,18.030384,18.030384,0.0


In [114]:
def prepare_group_scores(score_table):
    target_jobs = resume_master[["resume_id", "target_job_family"]]
    result = target_jobs.merge( score_table[["resume_id", "job_id", "ranking_score"]],
        left_on=["resume_id", "target_job_family"],
        right_on=["resume_id", "job_id"],
        how="left",
        validate="one_to_one")
    result = result.merge( protected_attrs, on="resume_id",how="left", validate="one_to_one" )
    assert len(result) == 185
    assert result["ranking_score"].notna().all()
    return result
group_scores_before = prepare_group_scores( baseline_rankings)
group_scores_after = prepare_group_scores( ranking_table)

In [115]:
group_results = []
for attribute in ["inferred_gender", "college_tier"]:
    before_means, before_gap = group_gap(group_scores_before["ranking_score"],group_scores_before[attribute])
    after_means, after_gap = group_gap( group_scores_after["ranking_score"],group_scores_after[attribute] )
    print("\nATTRIBUTE:", attribute)
    print("Before group averages:", before_means)
    print("After group averages:", after_means)
    print("Before group gap:", round(before_gap, 6))
    print("After group gap:", round(after_gap, 6))
    group_results.append({
        "attribute": attribute,
        "before_group_means": before_means,
        "before_group_gap": before_gap,
        "after_group_means": after_means,
        "after_group_gap": after_gap
    })


ATTRIBUTE: inferred_gender
Before group averages: {'Female': 42.971147098043986, 'Male': 41.140596374787606}
After group averages: {'Female': 42.97118437404619, 'Male': 41.147118454524694}
Before group gap: 1.830551
After group gap: 1.824066

ATTRIBUTE: college_tier
Before group averages: {'Tier 1': 43.65636326418361, 'Tier 2': 39.34866196625489, 'Tier 3': 43.2854505124244}
After group averages: {'Tier 1': 43.65736338861145, 'Tier 2': 39.34970179500223, 'Tier 3': 43.29474090222368}
Before group gap: 4.307701
After group gap: 4.307662


In [116]:
group_gap_results = pd.DataFrame(group_results)

In [117]:
group_gap_results

,attribute,before_group_means,before_group_gap,after_group_means,after_group_gap
0,inferred_gender,"{'Female': 42.971147098043986, 'Male': 41.1405...",1.830551,"{'Female': 42.97118437404619, 'Male': 41.14711...",1.824066
1,college_tier,"{'Tier 1': 43.65636326418361, 'Tier 2': 39.348...",4.307701,"{'Tier 1': 43.65736338861145, 'Tier 2': 39.349...",4.307662


In [118]:
human_labels = pd.read_excel( DATA_PATH,sheet_name="resume_job_labels")

In [119]:
def evaluate_screener(score_table, version):
    evaluation = score_table[ ["resume_id", "job_id", "ranking_score"]].merge(
        human_labels[[
                "resume_id",
                "job_id",
                "human_suitability_score",
                "shortlist"] ], on=["resume_id", "job_id"], how="inner",validate="one_to_one" )
    
    evaluation["shortlist_flag"] = (
        evaluation["shortlist"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("Y")
    )
    assert len(evaluation) == 555
    quality_rows = []
    for job_id, group in evaluation.groupby("job_id"):
        model_top5 = group.nlargest(5, "ranking_score")
        human_top5 = (
            group.sort_values(
                "human_suitability_score",
                ascending=False
            ).head(5)
        )
        hits = int(model_top5["shortlist_flag"].sum())
        total_shortlisted = int(group["shortlist_flag"].sum())
        precision = hits / 5
        recall = (hits / total_shortlisted
                  if total_shortlisted else 0)
        ndcg = ndcg_score( [group["human_suitability_score"].to_numpy()],  [group["ranking_score"].to_numpy()],  k=5)
        overlap = len( set(model_top5["resume_id"]) & set(human_top5["resume_id"]) )
        quality_rows.append({
            "version": version,
            "job_id": job_id,
            "model_top5_hits": hits,
            "precision_at_5_pct": precision * 100,
            "recall_at_5_pct": recall * 100,
            "ndcg_at_5": float(ndcg),
            "top5_overlap": overlap
        })
    quality = pd.DataFrame(quality_rows)
    total_hits = quality["model_top5_hits"].sum()
    total_shortlisted = evaluation["shortlist_flag"].sum()
    overall = {
        "version": version,
        "precision_at_5_pct": total_hits / 50 * 100,
        "recall_at_5_pct": total_hits / total_shortlisted * 100,
        "mean_ndcg_at_5": quality["ndcg_at_5"].mean(),
        "mean_top5_overlap": quality["top5_overlap"].mean()
    }
    return quality, overall

In [120]:
# Evaluate both versions

In [121]:
quality_before, metrics_before = evaluate_screener(baseline_rankings,"Before college-tier masking")
quality_after, metrics_after = evaluate_screener(ranking_table,"After college-tier masking")

In [122]:
ranking_quality_comparison = pd.DataFrame([metrics_before,metrics_after])

In [123]:
print("RANKING QUALITY COMPARISON")
display(ranking_quality_comparison.round(4))

RANKING QUALITY COMPARISON


,version,precision_at_5_pct,recall_at_5_pct,mean_ndcg_at_5,mean_top5_overlap
0,Before college-tier masking,94.0,49.4737,0.9839,3.6
1,After college-tier masking,94.0,49.4737,0.9839,3.6


In [95]:
job_quality_comparison = quality_before.merge( quality_after, on="job_id",suffixes=("_before", "_after"))

In [96]:
job_quality_comparison.round(4)

,version_before,job_id,model_top5_hits_before,precision_at_5_pct_before,recall_at_5_pct_before,ndcg_at_5_before,top5_overlap_before,version_after,model_top5_hits_after,precision_at_5_pct_after,recall_at_5_pct_after,ndcg_at_5_after,top5_overlap_after
0,Before college-tier masking,J01,5,100.0,71.4286,1.0000,4,After college-tier masking,5,100.0,71.4286,1.0000,4
1,Before college-tier masking,J02,2,40.0,66.6667,0.9544,4,After college-tier masking,2,40.0,66.6667,0.9544,4
2,Before college-tier masking,J03,5,100.0,41.6667,1.0000,2,After college-tier masking,5,100.0,41.6667,1.0000,2
3,Before college-tier masking,J04,5,100.0,31.2500,0.9352,2,After college-tier masking,5,100.0,31.2500,0.9352,2
4,Before college-tier masking,J05,5,100.0,55.5556,1.0000,4,After college-tier masking,5,100.0,55.5556,1.0000,4
5,Before college-tier masking,J06,5,100.0,83.3333,0.9950,5,After college-tier masking,5,100.0,83.3333,0.9950,5
6,Before college-tier masking,J07,5,100.0,62.5000,1.0000,4,After college-tier masking,5,100.0,62.5000,1.0000,4
7,Before college-tier masking,J08,5,100.0,55.5556,1.0000,2,After college-tier masking,5,100.0,55.5556,1.0000,2
8,Before college-tier masking,J09,5,100.0,35.7143,0.9572,4,After college-tier masking,5,100.0,35.7143,0.9572,4
9,Before college-tier masking,J10,5,100.0,45.4545,0.9969,5,After college-tier masking,5,100.0,45.4545,0.9969,5


In [129]:
from sklearn.metrics import ndcg_score
def evaluate_fairness_ranking(score_table, version):
    evaluation = score_table[
        ["resume_id", "job_id", "ranking_score"]].merge(human_labels[
            [
                "resume_id",
                "job_id",
                "human_suitability_score",
                "shortlist"
            ]
        ],
        on=["resume_id", "job_id"],
        how="inner",
        validate="one_to_one"
    )
    evaluation["shortlist_flag"] = (
        evaluation["shortlist"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("Y")
    )
    rows = []
    for job_id, group in evaluation.groupby("job_id"):
        model_top5 = group.sort_values(
            ["ranking_score", "resume_id"],
            ascending=[False, True],
            kind="mergesort"
        ).head(5)
        human_top5 = group.sort_values(
            ["human_suitability_score", "resume_id"],
            ascending=[False, True],
            kind="mergesort"
        ).head(5)
        hits = int(model_top5["shortlist_flag"].sum())
        total_shortlisted = int(group["shortlist_flag"].sum())
        ndcg = ndcg_score(
            [group["human_suitability_score"].to_numpy()],
            [group["ranking_score"].to_numpy()],
            k=5
        )
        overlap = len(
            set(model_top5["resume_id"])
            & set(human_top5["resume_id"])
        )
        rows.append({
            "version": version,
            "job_id": job_id,
            "model_top5_hits": hits,
            "precision_at_5_pct": hits / 5 * 100,
            "recall_at_5_pct": (
                hits / total_shortlisted * 100
                if total_shortlisted else 0
            ),
            "ndcg_at_5": float(ndcg),
            "top5_overlap": overlap
        })
    return pd.DataFrame(rows)

In [130]:
quality_before_final = evaluate_fairness_ranking(baseline_rankings,"Before correction")
quality_after_final = evaluate_fairness_ranking(ranking_table,"After correction")
final_quality_comparison = pd.concat([quality_before_final, quality_after_final],ignore_index=True)

In [131]:
final_quality_comparison

,version,job_id,model_top5_hits,precision_at_5_pct,recall_at_5_pct,ndcg_at_5,top5_overlap
0,Before correction,J01,5,100.0,71.428571,1.000000,3
1,Before correction,J02,2,40.0,66.666667,0.954447,3
2,Before correction,J03,5,100.0,41.666667,1.000000,2
3,Before correction,J04,5,100.0,31.250000,0.935169,3
4,Before correction,J05,5,100.0,55.555556,1.000000,4
5,Before correction,J06,5,100.0,83.333333,0.995022,5
6,Before correction,J07,5,100.0,62.500000,1.000000,4
7,Before correction,J08,5,100.0,55.555556,1.000000,2
8,Before correction,J09,5,100.0,35.714286,0.957203,4
9,Before correction,J10,5,100.0,45.454545,0.996947,4


In [126]:
twin_results.to_csv(EVIDENCE_DIR / "twin_gap_comparison.csv",index=False)
group_scores_before.to_csv(EVIDENCE_DIR / "group_scores_before.csv",index=False)
group_scores_after.to_csv(EVIDENCE_DIR / "group_scores_after.csv",index=False)
group_gap_results.to_csv(EVIDENCE_DIR / "group_gap_comparison.csv",index=False)
final_quality_comparison.to_csv(EVIDENCE_DIR / "ranking_quality_before_after.csv",index=False)

In [ ]:
## Task 12 – Resume Screener Fairness Audit
'''
I tested our resume screener using 25 pairs of resumes that had the same skills and experience but different gender-coded names, and in some cases, different college tiers.

Before and after removing college-tier information, the average and maximum score gaps between the twin resumes remained zero.

The gender group gap changed slightly from 1.830551 to 1.824066, while the college-tier gap changed from 4.307701 to 4.307662, showing that the improvement was very small.

Our ranking performance also remained unchanged, with 94% Precision@5, 49.47% Recall@5 and 0.9839 NDCG@5.

From this task, I understood that equal accuracy does not always mean equal treatment, and I believe fairness decisions should involve the organisation, recruiters and responsible AI teams, with ethical and legal considerations taking priority over small improvements in model performance.